# Data

In [1]:
planning_horizon = 480*60
directory = 'LOOSE_DUEDATE'
maxNumSet = 1
ReworkProbability = 0.03

import sys
import os
current_directory = os.getcwd()
parent_directory = os.path.abspath(os.path.join(current_directory, os.pardir))
sys.path.append(parent_directory)

import pandas as pd
import numpy as np
import random
import math
import pickle

# Load the Excel file
PlanMaster                 = pd.read_excel("DataMaster.xlsx", sheet_name="Plan")
ComponentMaster            = pd.read_excel("DataMaster.xlsx", sheet_name="ComponentMaster")

In [2]:
# from util.data_indentifier import MasterData
# # Master
# master              = {}
# all_machine_ids = [
#     int(machine_id)
#     for machines in ComponentMaster["MachineID"]
#     for machine_id in machines.split(",")
# ]
# K = len(set(all_machine_ids))
# I = 100


# for component_id, group in ComponentMaster.groupby("ComponentID"):
#     n_       = len(group)  # Number of operations
#     duration = group["BatchedDurationInSecond"].sum() # Duration
    
#     MC_i     = [[] for _ in range(I)]  # List of machines for each operation
#     n_MC_i   = [0 for _ in range(I)]   # Number of machines for each operation
#     p_ik     = np.full((I, K), 999)
#     h_ik     = np.zeros((I, K), dtype=int)
    
#     i = 0
#     # Iterate through operations
#     for _, row in group.iterrows():
#         machines = [int(m) - 1 for m in row["MachineID"].split(",")]
#         MC_i[i].extend(machines)
#         n_MC_i[i] += len(machines)
        
#         for k in machines:
#             p_ik[i, k] = row['BatchedDurationInSecond']
#             h_ik[i, k] = 1

#         i += 1
    
#     master[component_id] = MasterData(p_ik, h_ik, n_, MC_i, n_MC_i, duration)

# with open('ComponentMaster.pkl', 'wb') as f:
#     pickle.dump(master, f)

In [7]:
# for Set in range(1, 49):
SetList = ['_fixed_instance'] + [case+1 for case in range(1, 48)]
# SetList = [47]

for Set in SetList:
    Plan                       = PlanMaster[PlanMaster['Set'] == Set]
    JobToSchedule              = ComponentMaster[ComponentMaster['ComponentID'].isin(Plan['ComponentID'])].copy()
    JobToSchedule['MachineID'] = JobToSchedule['MachineID'].astype(str)
    OperationCount             = JobToSchedule['ComponentID'].value_counts().to_dict()
    K                          = 30

    if directory == "LOOSE_DUEDATE":
        JobList                = [component for _, row in Plan.iterrows() for component in [row['ComponentID']] * int(row['BatchedQuantity'])]
        J                      = len(JobList)
        Deadline               = [planning_horizon for j in range(J)]
    else:

        total_jobs             = Plan["BatchedQuantity"].sum()
        max_list_size          = int(0.03 * total_jobs)
        remaining_batches      = {row["ComponentID"]: row["BatchedQuantity"] for _, row in Plan.iterrows()}

        JobList = []
        JobList.extend(list(remaining_batches.keys()))

        for component_id in JobList:
            remaining_batches[component_id] -= 1

        remaining_slots = max_list_size - len(JobList)
        while remaining_slots > 0:
            available_components = [comp for comp, qty in remaining_batches.items() if qty > 0]
            selected_component = random.choice(available_components)

            JobList.append(selected_component)
            remaining_batches[selected_component] -= 1
            remaining_slots -= 1

        J                      = len(JobList)
        component_duration     = ComponentMaster.groupby('ComponentID')['BatchedDurationInSecond'].sum().to_dict()
        Deadline               = [int(math.ceil(component_duration[j] * random.choice([0.5, 1, 1.5]))) for j in JobList]


        # New Jobs
        remaining_info_file    = f'{directory}/Case{Set}_{planning_horizon // 60}_InfoNewJob.pkl'

        with open(remaining_info_file, 'wb') as f:
            pickle.dump(remaining_batches, f)

    # Ensure all selected components are present in the OperationCount
    for comp in JobList:
        if comp not in OperationCount:
            OperationCount[comp] = 0

    # Creating the text file in the specified format
    output_file = f'{directory}/Case{Set}_{planning_horizon // 60}.txt'

    with open(output_file, 'w') as f:
        f.write(f"{J}\n")
        f.write(f"{K}\n")
        f.write(" ".join(str(OperationCount[comp_id]) for comp_id in JobList) + "\n")
        f.write(" ".join(map(str, Deadline)) + "\n")
        job_id = 1
        for comp_id in JobList:
            comp_df = JobToSchedule[JobToSchedule['ComponentID'] == comp_id]
            for op_id, row in enumerate(comp_df.itertuples(), start=1):
                machines = row.MachineID.replace(' ', '').split(',')
                for machine in machines:
                    f.write(f"{job_id} {op_id} {machine} {row.BatchedDurationInSecond}\n")
            job_id += 1


print("Case generation complete.")


Case generation complete.


# Scenario

In [18]:
def get_info(file_path):
    machine_set = set()
    with open(file_path, 'r') as file:
        data = file.readlines()
        J = int(data[0].strip())
        for line in data[4:]:
            parts   = line.split()
            machine = int(parts[2])
            machine_set.add(machine)
    machine_list = list(machine_set)
    
    return J, machine_list

import ast
import math
import random
import pandas as pd
import numpy as np
from scipy.stats import weibull_min


# Read the Excel file
WeibullDistribution = pd.read_excel('DataMaster.xlsx', sheet_name='Distribution')


descriptions = {
                    'A': 'urgent',
                    'B': 'normal',
                    'C': 'loose'
                }

ScenarioList      = ['_fixed_scenario', 'A', 'B', 'C', 'D', 'E', 'F', 'G']
TypeList          = ['A', 'B', 'C']

Set = '_fixed_instance'
sourcename        = f'{directory}/Case{Set}_{planning_horizon // 60}.txt'
J, machine_list   = get_info(sourcename)

if directory == "TIGHT_DUEDATE":
    # Read remaining info + create list of remaining jobs
    remaining_info_file = f'{directory}/Case{Set}_{planning_horizon // 60}_InfoNewJob.pkl'
    with open(remaining_info_file, 'rb') as f:
        remaining_batches = pickle.load(f)

    new_jobs_indices = [comp_id for comp_id, qty in remaining_batches.items() for _ in range(qty)]
    random.shuffle(new_jobs_indices)

    # Read master
    with open('ComponentMaster.pkl', 'rb') as f:
        master = pickle.load(f)
    

for Scenario in ScenarioList:
    filename      = f'{directory}/Scenario{Scenario}_480.txt'
    # Create a text file to store the results
    with open(filename, 'w') as f:
        
        if directory == "LOOSE_DUEDATE": # =======================================================
            scenario_type = random.choice(TypeList)

            # Separate machine breakdown and defected jobs data
            f.write('Defected Jobs\n')
            f.write('JobID, Deadline_info, Description\n')
        
            # Determine description based on scenario
            description = descriptions[scenario_type]
            
            # Set deadline based on description
            if description == 'urgent':
                Deadline = np.random.randint(1, 11, size=J)
            elif description == 'normal':
                Deadline = np.full(J, planning_horizon)
            elif description == 'loose':
                Deadline = np.full(J, 2 * planning_horizon)
            
            # Generate defected jobs with 3% probability
            defected_jobs_indices = np.where(np.random.uniform(size=J) < ReworkProbability)[0] + 1  # One-indexed
            for job_id in defected_jobs_indices:
                deadline = Deadline[job_id - 1]  # Adjusting for one-indexing
                f.write(f'{job_id:<10}, {deadline:<10}, {description:<10}\n')

        else: #TIGHT DUEDATE" ==================================================================
            f.write('Remaining Jobs\n')
            f.write('ComponentID, \t Arrival_time, Deadline')

            arrival_time = 0

            for job_id in new_jobs_indices:
                component_profile = master[job_id]
                arrival_time     += random.choice([50, 100])
                duedate           = arrival_time + math.ceil(component_profile.duration*random.choice([0.5, 1, 1.5]))
                f.write(f'{job_id:<10},  \t {arrival_time:<10}, {duedate:<10}\n')
                
        f.write('\nMachineBreakdown\n')
        f.write('MachineID, BD_time, \tRe_time\n')
        # Iterate over each row in the DataFrame
        for _, row in WeibullDistribution.iterrows():
            machine_id = row['MachineID']
            if machine_id in machine_list:
                shape_up, loc_up, scale_up       = ast.literal_eval(row['ParameterUpTime'])
                shape_down, loc_down, scale_down = ast.literal_eval(row['ParameterDownTime'])
                
                # Calculate uptime and downtime
                t = -604800*2 # 14 days in seconds
                downtime = 0
                breakdown_time_list = []
                repairt_time_list = []
                while t < planning_horizon:
                    skip     = False
                    t       += downtime
                    uptime   = math.ceil(weibull_min.rvs(c=shape_up, loc=0,   scale=scale_up))*60 # change to second
                    count    = 0
                    while uptime < 300:
                        uptime   = math.ceil(weibull_min.rvs(c=shape_up, loc=0,   scale=scale_up))*60 # change to second
                        count += 1
                        if count == 10:
                            skip = True
                            break

                    downtime = math.ceil(weibull_min.rvs(c=shape_down, loc=0, scale=scale_down))*60 # change to second
                    t += uptime
                    if skip == True:
                        break
                    else:
                        if t > 0 and t < planning_horizon * 1.25:
                            breakdown_time_list.append(t)
                            repairt_time_list  .append(downtime)
            
            # Write the machine breakdown results to the text file
            for i in range(len(breakdown_time_list)):
                f.write(f'{machine_id:<10}{breakdown_time_list[i]:<15.1f}{repairt_time_list[i]:<15.1f}\n')

    print("Scenario generation complete.")


Scenario generation complete.
Scenario generation complete.
Scenario generation complete.
Scenario generation complete.
Scenario generation complete.
Scenario generation complete.
Scenario generation complete.
Scenario generation complete.


In [ ]:
print(machine_list)

In [ ]:
# def get_info(file_path):
#     machine_set = set()
#     with open(file_path, 'r') as file:
#         data = file.readlines()
#         J = int(data[0].strip())
#         for line in data[4:]:
#             parts   = line.split()
#             machine = int(parts[2])
#             machine_set.add(machine)
#     machine_list = list(machine_set)
    
#     return J, machine_list

# import ast
# import math
# import random
# import pandas as pd
# import numpy as np
# from scipy.stats import weibull_min


# # Read the Excel file
# WeibullDistribution = pd.read_excel('DataMaster.xlsx', sheet_name='Distribution')
# descriptions = {
#                     'A': 'urgent',
#                     'B': 'normal',
#                     'C': 'loose'
#                 }

# ScenarioList      = ['_fixed_scenario', 'A', 'B', 'C', 'D', 'E', 'F', 'G']
# TypeList          = ['A', 'B', 'C']

# Set = '_fixed_instance'
# sourcename        = f'{directory}/Case{Set}_{planning_horizon // 60}.txt'
# J, machine_list   = get_info("jobs_small.txt")

# for Scenario in ScenarioList:
#     scenario_type = random.choice(TypeList)
#     filename      = f'{directory}/Scenario{Scenario}_480.txt'
#     # Create a text file to store the results
#     with open(filename, 'w') as f:

#         # Separate machine breakdown and defected jobs data
#         f.write('Defected Jobs\n')
#         f.write('JobID, Deadline_info, Description\n')
    
#         # Determine description based on scenario
#         description = descriptions[scenario_type]
        
#         # Set deadline based on description
#         if description == 'urgent':
#             Deadline = np.random.randint(1, 11, size=J)
#         elif description == 'normal':
#             Deadline = np.full(J, planning_horizon)
#         elif description == 'loose':
#             Deadline = np.full(J, 2 * planning_horizon)
        
#         # Generate defected jobs with 3% probability
#         defected_jobs_indices = np.where(np.random.uniform(size=J) < ReworkProbability)[0] + 1  # One-indexed
#         for job_id in defected_jobs_indices:
#             deadline = Deadline[job_id - 1]  # Adjusting for one-indexing
#             f.write(f'{job_id:<10}, {deadline:<10}, {description:<10}\n')

#         f.write('\nMachine Breakdown\n')
#         f.write('MachineID, BD_time, \tRe_time\n')
#         # Iterate over each row in the DataFrame
#         for _, row in WeibullDistribution.iterrows():
#             machine_id = row['MachineID']
#             if machine_id in machine_list:
#                 shape_up, loc_up, scale_up       = ast.literal_eval(row['ParameterUpTime'])
#                 shape_down, loc_down, scale_down = ast.literal_eval(row['ParameterDownTime'])
                
#                 # Calculate uptime and downtime
#                 t = -604800*2
#                 downtime = 0
#                 breakdown_time_list = []
#                 repairt_time_list = []
#                 while t < planning_horizon:
#                     skip     = False
#                     t       += downtime
#                     uptime   = math.ceil(weibull_min.rvs(c=shape_up, loc=0,   scale=scale_up))*60 # change to second
#                     count    = 0
#                     while uptime < 300:
#                         uptime   = math.ceil(weibull_min.rvs(c=shape_up, loc=0,   scale=scale_up))*60 # change to second
#                         count += 1
#                         if count == 10:
#                             skip = True
#                             break

#                     downtime = math.ceil(weibull_min.rvs(c=shape_down, loc=0, scale=scale_down))*60 # change to second
#                     t += uptime
#                     if skip == True:
#                         break
#                     else:
#                         if t > 0 and t < planning_horizon:
#                             breakdown_time_list.append(t)
#                             repairt_time_list  .append(downtime)
            
#             # Write the machine breakdown results to the text file
#             for i in range(len(breakdown_time_list)):
#                 f.write(f'{machine_id:<10}{breakdown_time_list[i]:<15.1f}{repairt_time_list[i]:<15.1f}\n')

#     print("Scenario generation complete.")
